### Import

In [21]:
import os
from pathlib import Path
from dotenv import load_dotenv
import logging
import sys
import numpy as np
import pandas as pd
from numpy.typing import NDArray
from collections.abc import Sequence
from typing import Callable
from sklearn.preprocessing import MinMaxScaler
from dataclasses import dataclass

### Config Settings

In [31]:
BASE_DIR = Path("./notebooks/development.ipynb").resolve().parent
load_dotenv(BASE_DIR / ".env")


class Config:
    _raw_db_path = os.getenv("DATABASE_PATH", "./database/main_database.db")

    if _raw_db_path.startswith(("notebooks/", "./notebooks/")):
        _raw_db_path = _raw_db_path.replace("notebooks/", "", 1).lstrip("./")

    if Path(_raw_db_path).is_absolute():
        DB_FULL_PATH = Path(_raw_db_path)
    else:
        DB_FULL_PATH = BASE_DIR / _raw_db_path

    LOG_LEVEL = os.getenv("LOG_LEVEL", "INFO").upper()

    @classmethod
    def get_db_url(cls) -> str:
        return f"sqlite:///{cls.DB_FULL_PATH}"

    @classmethod
    def validate_db_exists(cls):
        if not cls.DB_FULL_PATH.exists():
            raise FileNotFoundError(
                f"Database file not found at {cls.DB_FULL_PATH}\n"
                f"Check your '.env' file or ensure the './database/' folder exists."
            )


### Main

In [34]:
print(f"BASE_DIR: {BASE_DIR}")
print(f"DB Path : {Config.DB_FULL_PATH}")re
print(f"Exists  : {Config.DB_FULL_PATH.RESexists()}")

BASE_DIR: D:\projects\msdl-jci\notebooks\notebooks
DB Path : D:\projects\msdl-jci\notebooks\notebooks\database\main_database.db
Exists  : False


### Constant

In [23]:
_LOG_FMT = "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s"

_DATE_FMT = "%Y-%m-%d %H:%M:%S"

MACRO_BASE_COLS = ["bi_rate", "inflation_rate", "usd_idr"]

CHANGE_LAGS = (21, 63, 126)

ZSCORE_WINDOWS = (126, 252)

EPSILON = 1e-9

MONTH_MAP = {
    "januari": "01",
    "jan": "01",
    "februari": "02",
    "feb": "02",
    "maret": "03",
    "mar": "03",
    "april": "04",
    "apr": "04",
    "mei": "05",
    "may": "05",
    "juni": "06",
    "jun": "06",
    "juli": "07",
    "jul": "07",
    "agustus": "08",
    "agu": "08",
    "aug": "08",
    "september": "09",
    "sep": "09",
    "oktober": "10",
    "okt": "10",
    "oct": "10",
    "november": "11",
    "nov": "11",
    "desember": "12",
    "des": "12",
    "dec": "12",
}

# Machine Learning / Modeling Hyperparameters (Aligned with Thesis Table 5)
ML_LOOK_BACK: int = int(os.getenv("ML_LOOK_BACK", "28"))
ML_PREDICTION_HORIZON: int = int(os.getenv("ML_PREDICTION_HORIZON", "5"))
ML_HIDDEN_LAYERS: tuple[int, ...] = (50, 25)
ML_MAX_ITER: int = int(os.getenv("ML_MAX_ITER", "500"))
ML_RANDOM_STATE: int = int(os.getenv("ML_RANDOM_STATE", "42"))

### Utility

In [13]:
def read_numeric_series(
        csv_path: str | Path,
        column_name: str,
        remove_symbol: str = "",
) -> NDArray[np.float64]:
    df = pd.read_csv(Path(csv_path))

    if column_name not in df.columns:
        raise KeyError(f"Column '{column_name}' not found. Available: {list(df.columns)}")

    series = df[column_name].astype(str).str.strip()
    if remove_symbol:
        series = series.str.replace(remove_symbol, "", regex=False).str.replace(",", "", regex=False)

    return pd.to_numeric(series, errors="raise").to_numpy(dtype=np.float64)

In [14]:
def configure_logging(level: int | str = logging.INFO) -> None:
    if isinstance(level, str):
        level = getattr(logging, level.upper(), logging.INFO)

    logging.basicConfig(
        level=level,
        format=_LOG_FMT,
        datefmt=_DATE_FMT,
        stream=sys.stdout,
        force=True,
    )


def get_logger(name: str) -> logging.Logger:
    return logging.getLogger(name)

In [17]:
def _safe_pct_change(series: pd.Series, lag: int) -> pd.Series:
    prev = series.shift(lag)
    return (series - prev) / (prev.abs() + EPSILON)


def _safe_zscore(series: pd.Series, window: int) -> pd.Series:
    min_periods = max(21, window // 4)
    roll = series.rolling(window, min_periods=min_periods)
    mu = roll.mean()
    sd = roll.std().replace(0.0, np.nan)
    return ((series - mu) / (sd + EPSILON)).fillna(0.0)


def add_macro_change_features(
        df: pd.DataFrame,
        lags: Sequence[int] | None = None,
        z_windows: Sequence[int] | None = None,
) -> pd.DataFrame:
    lags = lags or CHANGE_LAGS
    z_windows = z_windows or ZSCORE_WINDOWS

    result = df.copy().sort_values("Date").reset_index(drop=True)
    shifted = {col: result[col].shift(1) for col in MACRO_BASE_COLS if col in result.columns}

    feature_map: dict[str, Callable[[pd.Series], pd.Series]] = {}
    for col, series in shifted.items():
        for lag in lags:
            key = f"{col}_chg_{lag}d"
            feature_map[key] = lambda s=series, l=lag: _safe_pct_change(s, l)
        for w in z_windows:
            key = f"{col}_z{w}d"
            feature_map[key] = lambda s=series, w=w: _safe_zscore(s, w)

    for name, func in feature_map.items():
        if name not in result.columns:
            result[name] = func()

    if "bi_rate_chg_63d" in result.columns:
        result["rate_hiking"] = (result["bi_rate_chg_63d"] > 0.02).astype(float)
        result["rate_cutting"] = (result["bi_rate_chg_63d"] < -0.02).astype(float)
    if "inflation_rate" in result.columns:
        result["high_inflation"] = (result["inflation_rate"].shift(1) > 5.0).astype(float)
    if "usd_idr_chg_63d" in result.columns:
        result["idr_depreciating"] = (result["usd_idr_chg_63d"] > 0.02).astype(float)

    return result


ENGINEERED_MACRO_COLS = [
    *[f"bi_rate_chg_{l}d" for l in CHANGE_LAGS],
    *[f"inflation_rate_chg_{l}d" for l in CHANGE_LAGS],
    *[f"usd_idr_chg_{l}d" for l in CHANGE_LAGS],
    *[f"usd_idr_z{w}d" for w in ZSCORE_WINDOWS],
    "rate_hiking", "rate_cutting", "high_inflation", "idr_depreciating",
]

In [22]:
logger = get_logger(__name__)


@dataclass(slots=True)
class MultiSourceTensors:
    X_tech: np.ndarray
    X_macro: np.ndarray
    X_news: np.ndarray
    y: np.ndarray
    returns_5d: np.ndarray
    dates: np.ndarray
    close_prices: np.ndarray
    tech_feature_cols: list[str]


def _parse_id_date(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip().str.lower()
    for k, v in MONTH_MAP.items():
        s = s.str.replace(k, v, regex=False)
    return pd.to_datetime(s, errors="coerce", dayfirst=True)


def _calc_tech_indicators(df: pd.DataFrame) -> pd.DataFrame:
    c = df["Close"].astype(float)
    h, l, v = df["High"].astype(float), df["Low"].astype(float), df["Volume"].astype(float)

    # RSI-14
    d = c.diff()
    g, lo = d.clip(lower=0), -d.clip(upper=0)
    ag = g.ewm(alpha=1 / 14, min_periods=14, adjust=False).mean()
    al = lo.ewm(alpha=1 / 14, min_periods=14, adjust=False).mean()
    df["RSI_14"] = 100 - 100 / (1 + ag / (al + 1e-9))

    # MACD (12,26,9)
    e12 = c.ewm(span=12, adjust=False).mean()
    e26 = c.ewm(span=26, adjust=False).mean()
    df["MACD"] = e12 - e26
    df["MACD_Signal"] = df["MACD"].ewm(span=9, adjust=False).mean()

    # ATR-14
    pc = c.shift(1)
    tr = pd.concat([h - l, (h - pc).abs(), (l - pc).abs()], axis=1).max(axis=1)
    df["ATR_14"] = tr.ewm(alpha=1 / 14, min_periods=14, adjust=False).mean()

    # SMA-20
    df["SMA_20"] = c.rolling(20).mean()
    return df


class MultiSourceDatasetBuilder:
    TECH_COLS = ["Close", "Volume", "RSI_14", "MACD", "MACD_Signal", "ATR_14", "SMA_20"]
    MACRO_COLS = ["bi_rate", "inflation_rate", "usd_idr"]

    def __init__(self, look_back: int | None = None, horizon: int | None = None):
        cfg = Config()
        self.look_back = look_back or cfg.ML_LOOK_BACK
        self.horizon = horizon or cfg.ML_PREDICTION_HORIZON

    def _load_macro(self, bi_p, inf_p, kurs_p):
        bi = pd.read_csv(bi_p)
        bc = "BI-7Day-RR" if "BI-7Day-RR" in bi.columns else bi.columns[-1]
        bp = "Period" if "Period" in bi.columns else bi.columns[1]
        bi["Date"] = _parse_id_date(bi[bp])
        bi["bi_rate"] = bi[bc].astype(str).str.replace("%", "", regex=False).str.strip().astype(float)

        inf = pd.read_csv(inf_p)
        ic = "Data Inflasi" if "Data Inflasi" in inf.columns else inf.columns[-1]
        ip = "Periode" if "Periode" in inf.columns else inf.columns[0]
        inf["Date"] = _parse_id_date(inf[ip], day_first=False)
        inf["inflation_rate"] = inf[ic].astype(str).str.replace("%", "", regex=False).str.strip().astype(float)

        kur = pd.read_csv(kurs_p)
        kur["Date"] = pd.to_datetime(kur["Date"], errors="coerce")
        kur["usd_idr"] = pd.to_numeric(kur["Close"].astype(str).str.replace(",", "", regex=False), errors="coerce")

        return (
            bi.dropna(subset=["Date"]).sort_values("Date")[["Date", "bi_rate"]],
            inf.dropna(subset=["Date"]).sort_values("Date")[["Date", "inflation_rate"]],
            kur.dropna(subset=["Date"]).sort_values("Date")[["Date", "usd_idr"]],
        )

    def build_aligned_dataframe(self, jci_csv=None, bi_csv=None, inf_csv=None, kurs_csv=None, news_csv=None):
        cfg = Config()
        jci = pd.read_csv(jci_csv or cfg.JCI_HISTORICAL_CSV)
        jci["Date"] = pd.to_datetime(jci["Date"], errors="coerce")
        jci = jci.dropna(subset=["Date", "Close"]).sort_values("Date").reset_index(drop=True)
        _calc_tech_indicators(jci)

        bi, inf, kur = self._load_macro(
            bi_csv or cfg.BI_RATE_CSV,
            inf_csv or cfg.INFLATION_CSV,
            kurs_csv or cfg.KURS_CSV,
        )

        m = pd.merge(jci, kur, on="Date", how="left")
        m = pd.merge_asof(m.sort_values("Date"), bi.sort_values("Date"), on="Date", direction="backward")
        m = pd.merge_asof(m.sort_values("Date"), inf.sort_values("Date"), on="Date", direction="backward")

        for col in self.MACRO_COLS:
            m[col] = m[col].ffill()
        m = m.dropna(subset=self.MACRO_COLS).reset_index(drop=True)

        fc = m["Close"].shift(-self.horizon)
        m["return_5d"] = (fc - m["Close"]) / m["Close"]
        m["target_direction"] = (fc > m["Close"]).astype(float)
        m["future_close"] = fc

        news_p = news_csv or cfg.DAILY_NEWS_EMBEDDINGS_CSV
        emb_cols = [c for c in m.columns if c.startswith("emb_")]
        if not emb_cols:
            if Path(news_p).exists():
                ndf = pd.read_csv(news_p).rename(columns={"trade_date": "Date"})
                ndf["Date"] = pd.to_datetime(ndf["Date"], format="%Y-%m-%d", errors="coerce")
                ec = [c for c in ndf.columns if c.startswith("emb_")]
                m = pd.merge(m, ndf, on="Date", how="left")
                m[ec] = m[ec].fillna(0.0)
            else:
                ec = [f"emb_{i}" for i in range(cfg.NEWS_EMB_DIM)]
                m[ec] = 0.0
            emb_cols = ec
        else:
            m[emb_cols] = m[emb_cols].fillna(0.0)

        m = m.dropna(subset=self.TECH_COLS + ["future_close"]).reset_index(drop=True)
        logger.info("Aligned: %d samples (%s → %s)", len(m), m["Date"].min(), m["Date"].max())
        return m

    def create_multisource_tensors(self, df, feat_scaler=None, macro_scaler=None, fit_scalers=True, train_end_idx=None):
        if any(c in self.TECH_COLS for c in ("future_close", "return_5d", "target_direction")):
            raise ValueError("Target columns leaked into tech features")

        emb_cols = [c for c in df.columns if c.startswith("emb_")]
        raw_tech = df[self.TECH_COLS].values.astype(np.float32)
        raw_macro = df[self.MACRO_COLS].values.astype(np.float32)
        raw_news = df[emb_cols].values.astype(np.float32)
        lb = self.look_back
        n = len(df) - lb + 1
        if n <= 0:
            raise ValueError(f"Need ≥{lb} rows, got {len(df)}")

        if fit_scalers:
            feat_scaler = MinMaxScaler(feature_range=(0, 1))
            macro_scaler = MinMaxScaler(feature_range=(0, 1))
            end = train_end_idx or len(df)
            if train_end_idx and not (0 < train_end_idx <= len(df)):
                raise ValueError(f"train_end_idx={train_end_idx} out of range")
            feat_scaler.fit(raw_tech[:end])
            macro_scaler.fit(raw_macro[:end])
            logger.info("Scalers fit on [: %d] / %d rows", end, len(df))
        elif feat_scaler is None or macro_scaler is None:
            raise ValueError("Provide fitted scalers when fit_scalers=False")

        st = feat_scaler.transform(raw_tech)
        sm = macro_scaler.transform(raw_macro)

        # Vectorized sliding window via stride tricks (O(1) memory view)
        shape = (n, lb, len(self.TECH_COLS))
        strides = (st.strides[0], st.strides[0], st.strides[1])
        X_tech = np.lib.stride_tricks.as_strided(st, shape=shape, strides=strides).copy()

        idx = np.arange(lb - 1, len(df))
        tensors = MultiSourceTensors(
            X_tech=X_tech,
            X_macro=sm[idx],
            X_news=raw_news[idx],
            y=df["target_direction"].values[idx].astype(np.float32),
            returns_5d=df["return_5d"].values[idx].astype(np.float32),
            dates=df["Date"].dt.strftime("%Y-%m-%d").values[idx],
            close_prices=df["Close"].values[idx].astype(np.float32),
            tech_feature_cols=self.TECH_COLS,
        )
        return tensors, feat_scaler, macro_scaler